In [4]:
import os
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
try:
    from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
except ImportError:
    from langchain.chat_models import AzureChatOpenAI
    from langchain.embeddings import AzureOpenAIEmbeddings
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableSequence
from langchain.schema import BaseOutputParser
from langchain.callbacks import get_openai_callback
import json
import random
import warnings

# Suppress deprecation warnings for cleaner output
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Data Models
class MCQOption(BaseModel):
    option: str = Field(description="The option text")
    is_correct: bool = Field(description="Whether this option is correct")
    explanation: str = Field(description="Explanation for why this is correct/incorrect")

class MCQuestion(BaseModel):
    question: str = Field(description="The question text")
    difficulty_level: str = Field(description="beginner, intermediate, advanced, or expert")
    topic: str = Field(description="Specific topic being tested")
    options: List[MCQOption] = Field(description="List of 4 options")
    detailed_explanation: str = Field(description="Comprehensive explanation of the correct answer")
    experience_level_targeted: str = Field(description="Target experience level")
    cognitive_level: str = Field(description="remember, understand, apply, analyze, evaluate, create")

class QuestionBank(BaseModel):
    questions: List[MCQuestion] = Field(description="Generated questions")
    metadata: Dict[str, Any] = Field(description="Generation metadata")

# Experience Level Mapper
class ExperienceMapper:
    @staticmethod
    def map_experience_to_difficulty(years: int) -> Dict[str, Any]:
        if years <= 1:
            return {
                "primary_difficulty": "beginner",
                "secondary_difficulty": "intermediate",
                "cognitive_levels": ["remember", "understand", "apply"],
                "complexity_focus": "syntax, basic concepts, fundamental patterns"
            }
        elif years <= 3:
            return {
                "primary_difficulty": "intermediate", 
                "secondary_difficulty": "advanced",
                "cognitive_levels": ["understand", "apply", "analyze"],
                "complexity_focus": "best practices, common patterns, debugging"
            }
        elif years <= 7:
            return {
                "primary_difficulty": "advanced",
                "secondary_difficulty": "expert", 
                "cognitive_levels": ["apply", "analyze", "evaluate"],
                "complexity_focus": "optimization, architecture, edge cases"
            }
        else:
            return {
                "primary_difficulty": "expert",
                "secondary_difficulty": "expert",
                "cognitive_levels": ["analyze", "evaluate", "create"],
                "complexity_focus": "system design, performance, advanced patterns"
            }

# Simple Web Search Tool (Fallback)
class SimpleSearchTool:
    """Simplified search tool that doesn't rely on external APIs"""
    
    def run(self, query: str) -> str:
        # Return some basic search context for common technologies
        search_contexts = {
            "react": """
            React best practices 2024: Hooks, functional components, Context API, 
            performance optimization with memo and useMemo, state management patterns,
            testing with Jest and React Testing Library, TypeScript integration,
            server-side rendering with Next.js, concurrent features.
            """,
            "python": """
            Python best practices: Type hints, async/await, dataclasses, 
            context managers, decorators, list comprehensions, error handling,
            testing with pytest, virtual environments, package management.
            """,
            "javascript": """
            JavaScript modern features: ES6+ syntax, async/await, destructuring,
            modules, classes, arrow functions, template literals, spread operator,
            closures, prototypes, event loop understanding.
            """
        }
        
        for tech, context in search_contexts.items():
            if tech.lower() in query.lower():
                return context
        
        return "Current industry trends focus on best practices, performance optimization, and real-world application scenarios."
# Custom Output Parser for MCQ
class MCQOutputParser(BaseOutputParser):
    def parse(self, text: str) -> MCQuestion:
        try:
            # Remove any markdown formatting
            clean_text = text.strip().replace('```json', '').replace('```', '')
            
            # Try to extract JSON from the text
            start_idx = clean_text.find('{')
            end_idx = clean_text.rfind('}') + 1
            
            if start_idx != -1 and end_idx > start_idx:
                json_text = clean_text[start_idx:end_idx]
                data = json.loads(json_text)
                return MCQuestion(**data)
            else:
                return self._fallback_parse(text)
                
        except Exception as e:
            print(f"Parsing error: {e}")
            return self._fallback_parse(text)
    
    def _fallback_parse(self, text: str) -> MCQuestion:
        """Enhanced fallback parsing with better error handling"""
        return MCQuestion(
            question="Error parsing question - please check the response format",
            difficulty_level="intermediate",
            topic="parsing_error",
            options=[
                MCQOption(option="Option A", is_correct=False, explanation="Parsing failed"),
                MCQOption(option="Option B", is_correct=True, explanation="Correct by default"),
                MCQOption(option="Option C", is_correct=False, explanation="Parsing failed"),
                MCQOption(option="Option D", is_correct=False, explanation="Parsing failed")
            ],
            detailed_explanation="The question could not be parsed properly. Please check the LLM response format.",
            experience_level_targeted="unknown",
            cognitive_level="understand"
        )

# Main MCQ Generation System
class AdaptiveMCQGenerator:
    def __init__(self, azure_config: Optional[Dict[str, str]] = None):
        # Azure OpenAI configuration
        if azure_config:
            self.model_name = azure_config.get('model_name')
            self.api_endpoint = azure_config.get('api_endpoint')
            self.api_version = azure_config.get('api_version')
            self.api_key = azure_config.get('api_key')
        else:
            # Load from environment variables
            self.model_name = os.getenv('AZURE_OPENAI_MODEL')
            self.api_endpoint = os.getenv('AZURE_ENDPOINT')
            self.api_version = os.getenv('AZURE_API_VERSION')
            self.api_key = os.getenv('AZURE_API_KEY')
        
        # Validate configuration
        if not all([self.model_name, self.api_endpoint, self.api_version, self.api_key]):
            raise ValueError("Missing required Azure OpenAI configuration. Please set environment variables or pass azure_config.")
        
        # Initialize Azure OpenAI LLM with proper parameters
        try:
            self.llm = AzureChatOpenAI(
                azure_deployment=self.model_name,  # Changed from deployment_name
                model_name=self.model_name,
                azure_endpoint=self.api_endpoint,
                api_version=self.api_version,  # Changed from openai_api_version
                api_key=self.api_key,  # Changed from openai_api_key
                temperature=0.7,
                max_tokens=1000
            )
        except Exception as e:
            print(f"Error initializing Azure OpenAI LLM: {e}")
            # Fallback initialization with different parameter names
            try:
                self.llm = AzureChatOpenAI(
                    deployment_name=self.model_name,
                    azure_endpoint=self.api_endpoint,
                    openai_api_version=self.api_version,
                    openai_api_key=self.api_key,
                    temperature=0.7,
                    max_tokens=1000
                )
            except Exception as e2:
                raise ValueError(f"Failed to initialize Azure OpenAI: {e2}")
        
        # Initialize embeddings for potential future use (optional)
        try:
            self.embeddings = AzureOpenAIEmbeddings(
                azure_deployment=self.model_name,
                azure_endpoint=self.api_endpoint,
                api_version=self.api_version,
                api_key=self.api_key,
                chunk_size=1000  # Add chunk_size parameter
            )
        except Exception as e:
            print(f"Warning: Could not initialize embeddings: {e}")
            self.embeddings = None
        
        # Initialize search tool with fallback
        try:
            from langchain.tools import DuckDuckGoSearchRun
            self.search_tool = DuckDuckGoSearchRun()
        except Exception:
            print("⚠️  Using fallback search tool (DuckDuckGo not available)")
            self.search_tool = SimpleSearchTool()
        
        self.experience_mapper = ExperienceMapper()
        
        # Setup parsers
        self.mcq_parser = PydanticOutputParser(pydantic_object=MCQuestion)
        
        # Initialize prompts and chains
        self._setup_prompts()
        self._setup_chains()
    
    def _setup_prompts(self):
        # Knowledge Research Prompt
        self.research_prompt = PromptTemplate(
            input_variables=["technology", "experience_level", "specific_topics"],
            template="""
            Research the latest trends, best practices, and common interview topics for {technology}.
            Focus on content suitable for someone with {experience_level} experience level.
            Specific areas of interest: {specific_topics}
            
            Provide:
            1. Current industry standards and best practices
            2. Common pain points and challenges
            3. Advanced concepts that separate experienced developers
            4. Real-world scenarios and edge cases
            """
        )
        
        # MCQ Generation Prompt
        self.mcq_prompt = ChatPromptTemplate.from_template("""
        You are an expert technical interviewer specializing in creating challenging MCQ questions.
        
        Context:
        - Technology: {technology}
        - Candidate Experience: {years_experience} years
        - Target Difficulty: {difficulty_level}
        - Cognitive Level: {cognitive_level}
        - Focus Areas: {focus_areas}
        - Research Context: {research_context}
        
        Create a challenging multiple-choice question that:
        1. Tests deep understanding, not just memorization
        2. Has plausible distractors that would fool less experienced candidates
        3. Requires practical knowledge and experience
        4. Focuses on real-world scenarios
        5. Has exactly 4 options where only one is completely correct
        
        Make the incorrect options sophisticated - they should be:
        - Partially correct or common misconceptions
        - Technically sound but inappropriate for the context
        - Real alternatives that work in different scenarios
        
        Return your response as valid JSON matching this exact format:
        {{
            "question": "Your challenging question here",
            "difficulty_level": "{difficulty_level}",
            "topic": "specific topic being tested",
            "options": [
                {{"option": "Option A text", "is_correct": false, "explanation": "Why this is wrong"}},
                {{"option": "Option B text", "is_correct": true, "explanation": "Why this is correct"}},
                {{"option": "Option C text", "is_correct": false, "explanation": "Why this is wrong"}},
                {{"option": "Option D text", "is_correct": false, "explanation": "Why this is wrong"}}
            ],
            "detailed_explanation": "Comprehensive explanation of the concept and why the correct answer is right",
            "experience_level_targeted": "{years_experience} years",
            "cognitive_level": "{cognitive_level}"
        }}
        
        {format_instructions}
        """)
        
    def _setup_chains(self):
        # Modern LangChain approach using RunnableSequence
        # Research Chain
        self.research_chain = self.research_prompt | self.llm
        
        # MCQ Generation Chain using the modern syntax
        self.mcq_chain = self.mcq_prompt | self.llm
    
    def research_technology(self, technology: str, experience_level: str) -> str:
        """Research current trends and topics for the technology"""
        try:
            # Use search tool to get current information
            search_query = f"{technology} interview questions {experience_level} 2024 best practices"
            search_results = self.search_tool.run(search_query)
            
            # Process with LLM using modern invoke method
            research_result = self.research_chain.invoke({
                "technology": technology,
                "experience_level": experience_level,
                "specific_topics": search_results[:1000]  # Limit token usage
            })
            
            # Extract content from response
            if hasattr(research_result, 'content'):
                return research_result.content
            else:
                return str(research_result)
                
        except Exception as e:
            print(f"Research warning: {e}")
            return f"Focus on {technology} best practices, common patterns, and real-world scenarios for {experience_level} developers."
    
    def generate_question(
        self, 
        technology: str, 
        years_experience: int,
        specific_topic: Optional[str] = None
    ) -> MCQuestion:
        """Generate a single adaptive MCQ question"""
        
        # Map experience to difficulty parameters
        exp_mapping = self.experience_mapper.map_experience_to_difficulty(years_experience)
        
        # Choose difficulty level (80% primary, 20% secondary for variety)
        difficulty = exp_mapping["primary_difficulty"] if random.random() < 0.8 else exp_mapping["secondary_difficulty"]
        cognitive_level = random.choice(exp_mapping["cognitive_levels"])
        
        # Research current topics
        research_context = self.research_technology(technology, f"{years_experience} years")
        
        # Generate the question
        try:
            # Use modern invoke method instead of deprecated run
            response = self.mcq_chain.invoke({
                "technology": technology,
                "years_experience": years_experience,
                "difficulty_level": difficulty,
                "cognitive_level": cognitive_level,
                "focus_areas": exp_mapping["complexity_focus"],
                "research_context": research_context[:500],  # Limit context
                "format_instructions": self.mcq_parser.get_format_instructions()
            })
            
            # Extract content from response
            if hasattr(response, 'content'):
                response_text = response.content
            else:
                response_text = str(response)
            
            # Parse the response
            question = self.mcq_parser.parse(response_text)
            return question
            
        except Exception as e:
            print(f"Error generating question: {e}")
            # Return a more informative fallback question
            return self._generate_fallback_question(technology, difficulty, str(e))
    
    def _generate_fallback_question(self, technology: str, difficulty: str, error_msg: str = "") -> MCQuestion:
        """Generate a more informative fallback question if main generation fails"""
        
        # Create technology-specific fallback questions based on difficulty
        fallback_questions = {
            "react": {
                "beginner": {
                    "question": "Which React hook is used to manage component state?",
                    "options": [
                        ("useEffect", False, "useEffect is for side effects, not state management"),
                        ("useState", True, "useState is the primary hook for managing component state"),
                        ("useContext", False, "useContext is for consuming context values"),
                        ("useCallback", False, "useCallback is for memoizing functions")
                    ]
                },
                "intermediate": {
                    "question": "What happens when you call setState multiple times in the same render cycle?",
                    "options": [
                        ("Each call triggers a separate re-render", False, "React batches setState calls"),
                        ("Only the last setState call is applied", False, "All updates are applied, but batched"),
                        ("React batches the updates and triggers one re-render", True, "React automatically batches setState calls for performance"),
                        ("It causes an infinite loop", False, "setState batching prevents this issue")
                    ]
                },
                "advanced": {
                    "question": "In React 18, how does automatic batching differ from legacy batching?",
                    "options": [
                        ("It only works with class components", False, "Automatic batching works with all components"),
                        ("It batches updates in timeouts and promises", True, "React 18 extends batching to async operations"),
                        ("It requires manual flushSync calls", False, "Automatic batching works without manual intervention"),
                        ("It only batches useState updates", False, "It batches all state updates including useReducer")
                    ]
                }
            },
            "python": {
                "beginner": {
                    "question": "What is the correct way to handle exceptions in Python?",
                    "options": [
                        ("try/catch", False, "Python uses try/except, not try/catch"),
                        ("try/except", True, "try/except is the standard Python exception handling"),
                        ("handle/error", False, "This is not valid Python syntax"),
                        ("catch/throw", False, "Python uses different keywords")
                    ]
                }
            }
        }
        
        tech_lower = technology.lower()
        if tech_lower in fallback_questions and difficulty in fallback_questions[tech_lower]:
            fb_data = fallback_questions[tech_lower][difficulty]
            options = [
                MCQOption(option=opt[0], is_correct=opt[1], explanation=opt[2])
                for opt in fb_data["options"]
            ]
            
            return MCQuestion(
                question=fb_data["question"],
                difficulty_level=difficulty,
                topic=f"{technology}_fallback",
                options=options,
                detailed_explanation=f"This is a fallback {difficulty} question for {technology}. Error: {error_msg}",
                experience_level_targeted="fallback",
                cognitive_level="understand"
            )
        
        # Generic fallback
        return MCQuestion(
            question=f"What is an important concept when working with {technology}?",
            difficulty_level=difficulty,
            topic=f"{technology}_basics",
            options=[
                MCQOption(option="Understanding syntax only", is_correct=False, explanation="Syntax is just the beginning"),
                MCQOption(option="Best practices and patterns", is_correct=True, explanation="Best practices are crucial for professional development"),
                MCQOption(option="Memorizing documentation", is_correct=False, explanation="Understanding concepts is more important than memorization"),
                MCQOption(option="Using the latest features only", is_correct=False, explanation="Stability and compatibility matter too")
            ],
            detailed_explanation=f"This fallback question emphasizes the importance of best practices in {technology}. Original error: {error_msg}",
            experience_level_targeted="unknown",
            cognitive_level="understand"
        )
    
    def generate_question_set(
        self,
        technology: str,
        years_experience: int,
        num_questions: int = 5,
        topics: Optional[List[str]] = None
    ) -> QuestionBank:
        """Generate a set of adaptive questions"""
        
        questions = []
        metadata = {
            "technology": technology,
            "years_experience": years_experience,
            "target_difficulty": self.experience_mapper.map_experience_to_difficulty(years_experience),
            "generation_timestamp": "2024-01-01",  # You can use datetime.now()
            "total_questions": num_questions
        }
        
        with get_openai_callback() as cb:
            for i in range(num_questions):
                print(f"Generating question {i+1}/{num_questions}...")
                
                # Generate question
                question = self.generate_question(technology, years_experience)
                questions.append(question)
                
            metadata["token_usage"] = {
                "total_tokens": cb.total_tokens,
                "prompt_tokens": cb.prompt_tokens,
                "completion_tokens": cb.completion_tokens,
                "total_cost": cb.total_cost
            }
        
        return QuestionBank(questions=questions, metadata=metadata)
    
    def evaluate_question_quality(self, question: MCQuestion) -> Dict[str, Any]:
        """Evaluate the quality of a generated question"""
        evaluation_prompt = f"""
        Evaluate this MCQ question for quality and appropriateness:
        
        Question: {question.question}
        Options: {[opt.option for opt in question.options]}
        Target Experience: {question.experience_level_targeted}
        Difficulty: {question.difficulty_level}
        
        Rate on a scale of 1-10 for:
        1. Question clarity
        2. Option plausibility 
        3. Difficulty appropriateness
        4. Practical relevance
        5. Distractor quality
        
        Provide specific feedback for improvement.
        """
        
        try:
            # Use modern invoke method
            evaluation_response = self.llm.invoke(evaluation_prompt)
            evaluation_text = evaluation_response.content if hasattr(evaluation_response, 'content') else str(evaluation_response)
            return {"evaluation": evaluation_text, "score": "pending"}
        except Exception as e:
            return {"evaluation": f"Evaluation failed: {e}", "score": 0}

# Usage Example and Testing
def main():
    # Set up example environment variables (replace with your actual values)
    os.environ['AZURE_OPENAI_MODEL'] = 'gpt-4'  # Your deployment name
    os.environ['AZURE_ENDPOINT'] = 'https://your-resource.openai.azure.com/'
    os.environ['AZURE_API_VERSION'] = '2024-02-01'
    os.environ['AZURE_API_KEY'] = 'your-azure-api-key-here'
    
    try:
        # Initialize the system with Azure OpenAI
        generator = AdaptiveMCQGenerator()
        print("✅ Azure OpenAI initialized successfully!")
        
        # Test with a simple case first
        print("\n=== Testing with React - 5 years experience ===")
        question = generator.generate_question("React", 5)
        
        print(f"Question: {question.question}")
        print(f"Difficulty: {question.difficulty_level}")
        print(f"Topic: {question.topic}")
        print("\nOptions:")
        for i, opt in enumerate(question.options, 1):
            marker = "✅" if opt.is_correct else "❌"
            print(f"{i}. {opt.option} {marker}")
        
        print(f"\nExplanation: {question.detailed_explanation}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        print("\nTroubleshooting steps:")
        print("1. Make sure you have the correct LangChain version installed:")
        print("   pip install langchain-openai")
        print("2. Set your Azure OpenAI environment variables correctly")
        print("3. Verify your Azure OpenAI deployment is active")

# Simplified configuration validation
def validate_azure_config():
    """Validate that all required Azure OpenAI environment variables are set"""
    required_vars = {
        'AZURE_OPENAI_MODEL': 'Your Azure OpenAI deployment name',
        'AZURE_ENDPOINT': 'Your Azure OpenAI endpoint URL', 
        'AZURE_API_VERSION': 'Azure OpenAI API version (e.g., 2024-02-01)',
        'AZURE_API_KEY': 'Your Azure OpenAI API key'
    }
    
    missing_vars = []
    for var, description in required_vars.items():
        if not os.getenv(var):
            missing_vars.append(f"{var}: {description}")
    
    if missing_vars:
        print("❌ Missing required environment variables:")
        for var in missing_vars:
            print(f"   - {var}")
        print("\nSet them like this:")
        print("export AZURE_OPENAI_MODEL='your-deployment-name'")
        print("export AZURE_ENDPOINT='https://your-resource.openai.azure.com/'")
        print("export AZURE_API_VERSION='2024-02-01'")
        print("export AZURE_API_KEY='your-api-key'")
        return False
    
    print("✅ All Azure OpenAI environment variables are configured")
    return True

if __name__ == "__main__":
    # Only validate if running as main script
    if validate_azure_config():
        main()

✅ All Azure OpenAI environment variables are configured
✅ Azure OpenAI initialized successfully!

=== Testing with React - 5 years experience ===


d:\GEN_AI\langchain\venv\lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Research warning: Connection error.
Error generating question: Connection error.
Question: In React 18, how does automatic batching differ from legacy batching?
Difficulty: advanced
Topic: React_fallback

Options:
1. It only works with class components ❌
2. It batches updates in timeouts and promises ✅
3. It requires manual flushSync calls ❌
4. It only batches useState updates ❌

Explanation: This is a fallback advanced question for React. Error: Connection error.
